# 6-2 requires_grad와 Tensor gradient — 기본

직접 작성한 코드와 저장된 실행 결과를 정리했습니다.


In [ ]:
import random
import json
import math
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split

# 실습 결과가 매번 비슷하게 나오도록 seed를 고정합니다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [ ]:
x_plain = torch.tensor([1.0, 2.0, 3.0])
# TODO: gradient 추적이 켜진 Tensor를 만드세요.
x_train = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

out_plain = (x_plain * 2).sum()
out_train = (x_train * 2).sum()

print('plain grad_fn:', out_plain.grad_fn)
print('train grad_fn:', out_train.grad_fn)

plain grad_fn: None
train grad_fn: <SumBackward0 object at 0x7ac9e23599f0>


In [ ]:
base = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
feature = base * 3

# TODO: feature에서 계산 그래프를 끊으세요.
frozen_feature = feature.detach()

loss = frozen_feature.pow(2).mean()
try:
    loss.backward()
    print('base.grad:', base.grad)
except RuntimeError as e:
    print('그래프가 끊긴 값에서는 backward가 제한될 수 있습니다:', type(e).__name__)

그래프가 끊긴 값에서는 backward가 제한될 수 있습니다: RuntimeError


In [ ]:
model = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 1))

# TODO: 첫 번째 Linear layer의 파라미터를 freeze하세요.
for p in model[0].parameters():
    p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print('trainable params:', trainable)
print('frozen params:', frozen)

trainable params: 5
frozen params: 16
